# 01 · Base de conhecimento
Valida os 25 trechos ativos e as 11 fontes inventariadas (item 02 da pontuação).

In [1]:
import sys, os; sys.path.insert(0, os.path.abspath(".."))
import pandas as pd, json
from alertavida.nucleo import *

In [2]:
trechos = carregar_jsonl(RAIZ/'base'/'trechos.jsonl')
fontes = pd.read_csv(RAIZ/'base'/'fontes.csv')
df = pd.DataFrame(trechos)
assert df.id.is_unique, 'ids duplicados'
assert set(df.fonte_id) <= set(fontes.fonte_id), 'trecho com fonte_id ausente em fontes.csv'
assert not df.texto.str.startswith('[EXEMPLO]').any(), 'ainda há trechos de exemplo'
print(len(df), 'trechos |', df.texto.str.split().str.len().describe()[['min','mean','max']].round(0).to_dict())

25 trechos | {'min': 10.0, 'mean': 38.0, 'max': 81.0}


In [3]:
# Tabela resumida das fontes e trechos
tab = fontes.merge(df.groupby('fonte_id').size().rename('n_trechos'), on='fonte_id', how='left')
tab['n_trechos'] = tab.n_trechos.fillna(0).astype(int)
licencas = {'F01':'CC BY-ND 3.0','F02':'Domínio público','F03':'Ato normativo oficial',
            'F04':'Glossário; não licencia o PDF','F05':'Reprodução com crédito',
            'F06':'Regra interna do grupo','F07':'Institucional¹','F08':'Institucional¹',
            'F09':'Institucional¹','F10':'Institucional¹ · SC²','F11':'Institucional¹ · PR²'}
tab['licenca_resumida'] = tab.fonte_id.map(licencas)
tab[['titulo','formato','licenca_resumida','n_trechos']].rename(columns={'licenca_resumida':'licença','n_trechos':'trechos'})

,titulo,formato,licença,trechos
0,SAMU 192,HTML,CC BY-ND 3.0,9
1,Lei 9.503/1997 - Código de Trânsito Brasileiro...,HTML,Domínio público,7
2,Portaria GM/MS 2.048/2002 - Regulamento Técnic...,HTML,Ato normativo oficial,1
3,Plano de Dados Abertos 2024-2026 da PRF,PDF,Glossário; não licencia o PDF,0
4,Apresentação de números de emergência em rodov...,PDF,Reprodução com crédito,0
5,Critério de priorização do AlertaVida,Texto,Regra interna do grupo,1
6,Como usar o SAMU 192,HTML,Institucional¹,1
7,Acidentes em rodovias federais - perguntas fre...,HTML,Institucional¹,3
8,Telefone de emergência da PRF,HTML,Institucional¹,1
9,Emergência com produtos perigosos,HTML,Institucional¹ · SC²,1
